# 02 — Genomic QC

Quality control of DArTseq genotyping data for AnoKin.

**Data files**:
- `Report_DMo25-3087_SNP_2.csv` / `_SNP_3.csv` — SNP genotypes (paired ref/alt rows, binary 1/0/-)
- `Report_DMo25-3087_SNP_mapping_2.csv` / `_mapping_3.csv` — Read count data per allele
- `Report_DMo25-3087_SilicoDArT_1.csv` — Presence/absence markers

**Approach**:
1. Parse SNP paired rows into standard 0/1/2 genotype matrix
2. Per-sample QC: call rate, heterozygosity, read depth
3. Per-marker QC: call rate, MAF, reproducibility
4. Apply filters and report retained markers/samples

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

DATA = Path("../data")
metadata = pd.read_csv(DATA / "master_metadata.csv")

## 1. Parse DArTseq SNP data into genotype matrix

SNP files have paired rows (ref allele, alt allele). For each locus we combine them:
- `ref=1, alt=0` → **0** (homozygous reference)
- `ref=1, alt=1` → **1** (heterozygous)
- `ref=0, alt=1` → **2** (homozygous alternative)
- `ref=0, alt=0` → treat as missing
- `ref=-, alt=-` → missing (NaN)

In [2]:
def load_dart_snp(path, n_meta_cols=24):
    """Load a DArTseq SNP file and convert paired rows to 0/1/2 genotype matrix.
    
    Returns:
        geno: DataFrame (loci × samples) with values 0, 1, 2, or NaN
        marker_meta: DataFrame with marker-level metadata (one row per locus)
    """
    print(f"Loading {path.name}...")
    raw = pd.read_csv(path, header=6, low_memory=False)
    
    meta_cols = raw.columns[:n_meta_cols]
    sample_cols = raw.columns[n_meta_cols:]
    print(f"  {len(raw)} rows, {len(sample_cols)} samples, {len(meta_cols)} metadata cols")
    
    # Split into metadata and genotype matrices
    marker_meta_raw = raw[list(meta_cols)].copy()
    geno_raw = raw[list(sample_cols)].copy()
    
    # Replace '-' with NaN, convert to float
    geno_raw = geno_raw.replace("-", np.nan).astype(float)
    
    # Paired rows: even indices = ref allele, odd indices = alt allele
    n_loci = len(raw) // 2
    ref_idx = np.arange(0, n_loci * 2, 2)
    alt_idx = np.arange(1, n_loci * 2, 2)
    
    ref = geno_raw.iloc[ref_idx].values  # (n_loci, n_samples)
    alt = geno_raw.iloc[alt_idx].values
    
    # Combine: ref=1,alt=0 → 0; ref=1,alt=1 → 1; ref=0,alt=1 → 2; both NaN or 0,0 → NaN
    geno = np.full_like(ref, np.nan)
    both_called = ~np.isnan(ref) & ~np.isnan(alt)
    
    geno[both_called & (ref == 1) & (alt == 0)] = 0  # hom ref
    geno[both_called & (ref == 1) & (alt == 1)] = 1  # het
    geno[both_called & (ref == 0) & (alt == 1)] = 2  # hom alt
    geno[both_called & (ref == 0) & (alt == 0)] = np.nan  # double-null → missing
    
    # Use AlleleID from ref rows as unique locus identifier (CloneID is NOT unique)
    locus_ids = marker_meta_raw["AlleleID"].iloc[ref_idx].values
    
    geno_df = pd.DataFrame(geno, index=locus_ids, columns=sample_cols)
    geno_df.index.name = "AlleleID"
    
    # Marker metadata from ref rows only
    marker_meta = marker_meta_raw.iloc[ref_idx].copy()
    marker_meta.index = locus_ids
    
    print(f"  → {n_loci} loci × {len(sample_cols)} samples")
    print(f"  Unique locus IDs: {len(set(locus_ids))}")
    return geno_df, marker_meta

In [3]:
geno2, meta2 = load_dart_snp(DATA / "Report_DMo25-3087_SNP_2.csv")
geno3, meta3 = load_dart_snp(DATA / "Report_DMo25-3087_SNP_3.csv")

# Check for overlapping AlleleIDs
overlap = set(geno2.index) & set(geno3.index)
print(f"\nOverlapping AlleleIDs between SNP_2 and SNP_3: {len(overlap)}")

# Concatenate (drop overlaps from SNP_3 if any)
if overlap:
    geno3_unique = geno3.loc[~geno3.index.isin(overlap)]
    meta3_unique = meta3.loc[~meta3.index.isin(overlap)]
else:
    geno3_unique = geno3
    meta3_unique = meta3

geno = pd.concat([geno2, geno3_unique])
marker_meta = pd.concat([meta2, meta3_unique])

# Verify index uniqueness
assert geno.index.is_unique, f"Index not unique! {geno.index.duplicated().sum()} duplicates"
print(f"\nCombined: {geno.shape[0]} loci × {geno.shape[1]} samples")

Loading Report_DMo25-3087_SNP_2.csv...


  126188 rows, 894 samples, 24 metadata cols


  → 63094 loci × 894 samples
  Unique locus IDs: 63094
Loading Report_DMo25-3087_SNP_3.csv...


  101888 rows, 894 samples, 24 metadata cols


  → 50944 loci × 894 samples
  Unique locus IDs: 50944

Overlapping AlleleIDs between SNP_2 and SNP_3: 0



Combined: 114038 loci × 894 samples


## 2. Per-sample QC

In [4]:
# Per-sample statistics
n_loci = geno.shape[0]
sample_stats = pd.DataFrame({
    "n_called": geno.notna().sum(axis=0),
    "n_missing": geno.isna().sum(axis=0),
    "call_rate": geno.notna().mean(axis=0),
    "n_het": (geno == 1).sum(axis=0),
    "n_hom_ref": (geno == 0).sum(axis=0),
    "n_hom_alt": (geno == 2).sum(axis=0),
})
sample_stats["het_rate"] = sample_stats["n_het"] / sample_stats["n_called"]
sample_stats.index.name = "sample_id"

# Join to metadata for species labels
meta_lookup = metadata.drop_duplicates("sample_id").set_index("sample_id")[["morph_id", "mosquito_sex"]]
sample_stats = sample_stats.join(meta_lookup, how="left")

print(f"Per-sample statistics ({len(sample_stats)} samples):")
print(sample_stats[["call_rate", "het_rate", "n_called"]].describe().round(4))

Per-sample statistics (894 samples):
       call_rate  het_rate    n_called
count   894.0000  894.0000    894.0000
mean      0.6694    0.0300  76333.7204
std       0.1150    0.0055  13108.9756
min       0.0147    0.0022   1681.0000
25%       0.6716    0.0292  76585.0000
50%       0.6998    0.0312  79799.5000
75%       0.7173    0.0325  81794.7500
max       0.8444    0.0660  96298.0000


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Call rate distribution
axes[0].hist(sample_stats["call_rate"], bins=50, edgecolor="black", alpha=0.7)
axes[0].axvline(0.5, color="red", linestyle="--", label="0.5 threshold")
axes[0].set_xlabel("Call rate")
axes[0].set_ylabel("Number of samples")
axes[0].set_title("Per-sample call rate")
axes[0].legend()

# Heterozygosity distribution
axes[1].hist(sample_stats["het_rate"], bins=50, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Heterozygosity")
axes[1].set_ylabel("Number of samples")
axes[1].set_title("Per-sample heterozygosity")

# Call rate vs heterozygosity, coloured by species
for sp, grp in sample_stats.groupby("morph_id"):
    label = sp.replace("anopheles_", "An. ") if pd.notna(sp) else "unknown"
    axes[2].scatter(grp["call_rate"], grp["het_rate"], alpha=0.5, s=15, label=label)
axes[2].set_xlabel("Call rate")
axes[2].set_ylabel("Heterozygosity")
axes[2].set_title("Call rate vs heterozygosity")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(DATA / "../results/fig_sample_qc.png", bbox_inches="tight")
plt.show()

In [6]:
# Flag low-quality samples
SAMPLE_CR_THRESHOLD = 0.5
low_cr_samples = sample_stats[sample_stats["call_rate"] < SAMPLE_CR_THRESHOLD]
print(f"Samples with call rate < {SAMPLE_CR_THRESHOLD}: {len(low_cr_samples)}")
if len(low_cr_samples) > 0:
    print(low_cr_samples[["call_rate", "het_rate", "morph_id"]].sort_values("call_rate").head(20))

Samples with call rate < 0.5: 58
              call_rate  het_rate            morph_id
sample_id                                            
ep0000920222   0.014741  0.003569  anopheles_funestus
ep0000917412   0.014749  0.003567  anopheles_funestus
ep0000844088   0.035146  0.012974  anopheles_funestus
ep0000914960   0.037312  0.005640  anopheles_funestus
ep0000919855   0.043845  0.005200  anopheles_funestus
ep0000843782   0.043994  0.005382  anopheles_funestus
ep0000845424   0.044994  0.005652  anopheles_funestus
ep0000919589   0.045432  0.009458  anopheles_funestus
ep0000915364   0.050553  0.007112  anopheles_funestus
ep0000843308   0.052158  0.006725  anopheles_funestus
ep0000915042   0.054657  0.006417  anopheles_funestus
ep0000842382   0.060085  0.004816  anopheles_funestus
ep0000915906   0.065013  0.003102  anopheles_funestus
ep0000920045   0.098616  0.002223  anopheles_funestus
ep0000922533   0.122249  0.006814  anopheles_funestus
ep0000842808   0.183421  0.009131  anopheles_fune

## 3. Per-marker QC

In [7]:
# Per-marker statistics
n_samples = geno.shape[1]
marker_stats = pd.DataFrame({
    "call_rate": geno.notna().mean(axis=1),
    "n_called": geno.notna().sum(axis=1),
    "het_rate": (geno == 1).sum(axis=1) / geno.notna().sum(axis=1),
    "freq_alt": geno.sum(axis=1) / (2 * geno.notna().sum(axis=1)),  # allele freq of alt
})

# MAF = min(freq_alt, 1 - freq_alt)
marker_stats["maf"] = marker_stats["freq_alt"].apply(lambda f: min(f, 1 - f) if pd.notna(f) else np.nan)

# Add DArT-provided QC columns from marker metadata
for col in ["CallRate", "RepAvg", "AvgPIC", "AvgCountRef", "AvgCountSnp"]:
    if col in marker_meta.columns:
        marker_stats[f"dart_{col}"] = marker_meta[col].astype(float).values

print(f"Per-marker statistics ({len(marker_stats)} markers):")
print(marker_stats[["call_rate", "maf", "het_rate"]].describe().round(4))

Per-marker statistics (114038 markers):
         call_rate          maf     het_rate
count  114038.0000  114038.0000  114038.0000
mean        0.6694       0.0425       0.0255
std         0.2576       0.0860       0.0577
min         0.2002       0.0006       0.0000
25%         0.4318       0.0021       0.0000
50%         0.7148       0.0066       0.0034
75%         0.9228       0.0364       0.0215
max         0.9955       0.5000       0.6920


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Marker call rate
axes[0, 0].hist(marker_stats["call_rate"], bins=50, edgecolor="black", alpha=0.7)
axes[0, 0].axvline(0.7, color="red", linestyle="--", label="0.7 threshold")
axes[0, 0].set_xlabel("Call rate")
axes[0, 0].set_ylabel("Number of markers")
axes[0, 0].set_title("Per-marker call rate")
axes[0, 0].legend()

# MAF distribution
axes[0, 1].hist(marker_stats["maf"].dropna(), bins=50, edgecolor="black", alpha=0.7)
axes[0, 1].axvline(0.01, color="red", linestyle="--", label="MAF=0.01")
axes[0, 1].axvline(0.05, color="orange", linestyle="--", label="MAF=0.05")
axes[0, 1].set_xlabel("Minor allele frequency")
axes[0, 1].set_ylabel("Number of markers")
axes[0, 1].set_title("MAF distribution")
axes[0, 1].legend()

# Heterozygosity per marker
axes[0, 2].hist(marker_stats["het_rate"].dropna(), bins=50, edgecolor="black", alpha=0.7)
axes[0, 2].set_xlabel("Heterozygosity")
axes[0, 2].set_ylabel("Number of markers")
axes[0, 2].set_title("Per-marker heterozygosity")

# DArT reproducibility 
if "dart_RepAvg" in marker_stats.columns:
    axes[1, 0].hist(marker_stats["dart_RepAvg"].dropna(), bins=50, edgecolor="black", alpha=0.7)
    axes[1, 0].axvline(0.95, color="red", linestyle="--", label="0.95 threshold")
    axes[1, 0].set_xlabel("Reproducibility (RepAvg)")
    axes[1, 0].set_ylabel("Number of markers")
    axes[1, 0].set_title("DArT reproducibility")
    axes[1, 0].legend()

# Call rate vs MAF
axes[1, 1].scatter(marker_stats["call_rate"], marker_stats["maf"], alpha=0.05, s=1)
axes[1, 1].set_xlabel("Call rate")
axes[1, 1].set_ylabel("MAF")
axes[1, 1].set_title("Call rate vs MAF")

# DArT PIC
if "dart_AvgPIC" in marker_stats.columns:
    axes[1, 2].hist(marker_stats["dart_AvgPIC"].dropna(), bins=50, edgecolor="black", alpha=0.7)
    axes[1, 2].set_xlabel("PIC (Polymorphism Information Content)")
    axes[1, 2].set_ylabel("Number of markers")
    axes[1, 2].set_title("DArT PIC distribution")

plt.tight_layout()
plt.savefig(DATA / "../results/fig_marker_qc.png", bbox_inches="tight")
plt.show()

## 4. Genome mapping coverage

Check how markers distribute across chromosomes — do we have genome-wide coverage?

In [9]:
# Parse chromosome from the Chrom column
chrom_col = [c for c in marker_meta.columns if c.startswith("Chrom_")][0]
pos_col = [c for c in marker_meta.columns if c.startswith("ChromPosSnp") or c.startswith("ChromPosTag")][0]

def parse_chrom(val):
    if pd.isna(val) or val == "" or val == 0:
        return "unmapped"
    s = str(val)
    if "AfunGA1_" in s:
        return s.split("AfunGA1_")[1].split(" ")[0]
    return "unmapped"

marker_stats["chrom"] = marker_meta[chrom_col].apply(parse_chrom)
marker_stats["pos"] = pd.to_numeric(marker_meta[pos_col], errors="coerce")

chrom_counts = marker_stats["chrom"].value_counts()
print("Markers per chromosome:")
print(chrom_counts.to_string())
print(f"\nMapped to genome: {(marker_stats['chrom'] != 'unmapped').sum()}")
print(f"Unmapped: {(marker_stats['chrom'] == 'unmapped').sum()}")

Markers per chromosome:
chrom
2RL         39994
3RL         37645
unmapped    29540
X            6857
MT              2

Mapped to genome: 84498
Unmapped: 29540


## 5. Apply filters and export filtered genotype matrix

In [10]:
# Filter thresholds
MARKER_CR = 0.7       # minimum marker call rate
SAMPLE_CR = 0.5       # minimum sample call rate
MAF_MIN = 0.01        # minimum minor allele frequency
REPAVG_MIN = 0.95     # minimum DArT reproducibility (if available)

print("=== FILTERING ===")
print(f"Starting: {geno.shape[0]} markers × {geno.shape[1]} samples\n")

# Step 1: Remove low-quality samples
good_samples = sample_stats[sample_stats["call_rate"] >= SAMPLE_CR].index
geno_f = geno[good_samples].copy()
print(f"After sample filter (CR >= {SAMPLE_CR}): {geno_f.shape[1]} samples "
      f"(removed {geno.shape[1] - geno_f.shape[1]})")

# Step 2: Remove low call-rate markers (recalculate on remaining samples)
marker_cr = geno_f.notna().mean(axis=1)
keep_cr = marker_cr >= MARKER_CR
n_before = geno_f.shape[0]
geno_f = geno_f.loc[keep_cr]
print(f"After marker CR filter (>= {MARKER_CR}): {geno_f.shape[0]} markers "
      f"(removed {n_before - geno_f.shape[0]})")

# Step 3: MAF filter (recalculate on remaining data)
af = geno_f.sum(axis=1) / (2 * geno_f.notna().sum(axis=1))
maf_vals = af.apply(lambda f: min(f, 1-f))
keep_maf = maf_vals >= MAF_MIN
n_before = geno_f.shape[0]
geno_f = geno_f.loc[keep_maf]
print(f"After MAF filter (>= {MAF_MIN}): {geno_f.shape[0]} markers "
      f"(removed {n_before - geno_f.shape[0]})")

# Step 4: Reproducibility filter (if available)
if "dart_RepAvg" in marker_stats.columns:
    rep_vals = marker_stats.reindex(geno_f.index)["dart_RepAvg"]
    keep_rep = rep_vals >= REPAVG_MIN
    n_before = geno_f.shape[0]
    geno_f = geno_f.loc[keep_rep.fillna(False)]
    print(f"After reproducibility filter (>= {REPAVG_MIN}): {geno_f.shape[0]} markers "
          f"(removed {n_before - geno_f.shape[0]})")

print(f"\n=== FINAL: {geno_f.shape[0]} markers × {geno_f.shape[1]} samples ===")

=== FILTERING ===
Starting: 114038 markers × 894 samples



After sample filter (CR >= 0.5): 836 samples (removed 58)
After marker CR filter (>= 0.7): 62278 markers (removed 51760)


After MAF filter (>= 0.01): 21879 markers (removed 40399)
After reproducibility filter (>= 0.95): 19930 markers (removed 1949)

=== FINAL: 19930 markers × 836 samples ===


In [11]:
# Save filtered genotype matrix for downstream analysis
geno_f.to_csv(DATA / "geno_filtered.csv")
print(f"Saved filtered genotypes to data/geno_filtered.csv")
print(f"Shape: {geno_f.shape}")
print(f"Missing rate in filtered data: {geno_f.isna().mean().mean():.3f}")

Saved filtered genotypes to data/geno_filtered.csv
Shape: (19930, 836)
Missing rate in filtered data: 0.076
